# Run A3: written answers

**Purpose.** Test the Probability Anchor at each token of a written answer.

**Data.** The 420 Compound Prompts of Run A0, in two option orders.

**Method.** Qwen2.5-1.5B-Instruct writes an answer of up to 30 tokens. At each token: `(1 − strength) × log-probabilities of the Compound Prompt + strength × log-probabilities of the prompt with Ê`. Strengths: 0, 0.5 and 1. Two more answers are written for comparison: with no context, and with the evidence sentence alone.

**Score.** A rule reads the answer letter (A or B) in the first 80 characters of the answer. No letter counts as not correct.

**Criteria (set before the run, for each tagger training)**
8. Share of the drop recovered at strength 1 is at least 90%. Share recovered = (correct at strength 1 − correct at strength 0) / (correct with the evidence alone − correct at strength 0).
9. At most 5% of the answers are correct at strength 0 and not correct at strength 1.

**Output.** `clean_A/results/A3_*.csv`

## 1. Settings, the data of Run A0 and the subject model

In [ ]:
import re
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

SEEDS = (0, 1, 2)               # trainings of the tagger (Run A0)
SUBJECT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
MAX_NEW_TOKENS = 30
THRESHOLD = 0.90
LETTER_SLOT = "[LETTER]"

DRIVE_HOME = Path("/content/drive/MyDrive")
if not DRIVE_HOME.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = DRIVE_HOME / "selective-probability-anchoring"
CLEAN = ROOT / "clean_A"
OUT_DIR = CLEAN / "results"
assert (CLEAN / "e_hat.csv").exists(), "Run `Run A0 - data and Pressure Stripper` first"
OUT_DIR.mkdir(exist_ok=True)
pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_rows", 200)

items = pd.read_csv(CLEAN / "items.csv").set_index("item_id")
e_hat_table = pd.read_csv(CLEAN / "e_hat.csv")
e_hat_table["e_hat"] = e_hat_table["e_hat"].fillna("").astype(str)
clean_of = {(row.seed, row.key): row.e_hat for row in e_hat_table.itertuples()}   # (tagger training, item_id or wording name) -> Ê
print("Compound Prompts:", len(items), "| questions:", items["source_item_id"].nunique(), "| Ê:", len(clean_of))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(SUBJECT_MODEL, revision=MODEL_REVISION)
mdl = AutoModelForCausalLM.from_pretrained(SUBJECT_MODEL, revision=MODEL_REVISION,
                                           torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE).eval()


## 2. The anchor at each token

In [ ]:
def render_open(item, order, context=""):
    # the chat format of the other notebooks, with a free-text instruction
    a, b = ((item["correct_answer"], item["incorrect_answer"]) if order == "fwd"
            else (item["incorrect_answer"], item["correct_answer"]))
    question = f"Question: {item['question']}\nA. {a}\nB. {b}\n\nGive your answer and explain briefly."
    user = f"{context}\n\n{question}" if context else question
    return f"<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n"


written = {}      # (prompt of the Compound Prompt, clean prompt, strength) -> text; equal inputs are written one time


@torch.no_grad()
def generate_anchored(prompt_full, prompt_clean, strength):
    # at each token: (1 - strength) x log-probabilities of the full prompt + strength x log-probabilities of the clean prompt
    key = (prompt_full if strength != 1 else "", prompt_clean if strength != 0 else "", strength)
    if key in written:
        return written[key]
    prompts = [prompt_full] if strength == 0 else [prompt_clean] if strength == 1 else [prompt_full, prompt_clean]
    ids = [tok(prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(DEVICE) for prompt in prompts]
    past = [None] * len(prompts)
    tokens = []
    for _ in range(MAX_NEW_TOKENS):
        logps = []
        for k in range(len(prompts)):
            out = mdl(ids[k], past_key_values=past[k], use_cache=True)
            past[k] = out.past_key_values
            logps.append(F.log_softmax(out.logits[:, -1, :].float(), dim=-1))
        combined = logps[0] if len(logps) == 1 else (1 - strength) * logps[0] + strength * logps[1]
        next_id = combined.argmax(-1, keepdim=True)
        if next_id.item() == tok.eos_token_id:
            break
        tokens.append(next_id.item())
        ids = [next_id] * len(prompts)       # the same new token goes to both texts
    written[key] = tok.decode(tokens, skip_special_tokens=True)
    return written[key]


def letter_of(text):
    # the answer letter at the start of the text, or "?" if there is none
    found = re.search(r"\b(?:answer is|answer:|option|choice)?\s*\**\s*\(?([AB])\)?[.*:,\s]", str(text)[:80])
    return found.group(1) if found else "?"


## 3. Answers for each Compound Prompt, option order and tagger training

In [ ]:
rows = []
for n, (item_id, item) in enumerate(items.iterrows()):
    for order in ("fwd", "swap"):
        correct, incorrect = ("A", "B") if order == "fwd" else ("B", "A")
        prompt_full = render_open(item, order, item["mixed"].replace(LETTER_SLOT, incorrect))
        answers = [("all", "no context", generate_anchored(render_open(item, order), "", 0)),
                   ("all", "evidence sentence", generate_anchored(render_open(item, order, item["evidence"]), "", 0)),
                   ("all", "strength 0", generate_anchored(prompt_full, "", 0))]
        for seed in SEEDS:
            prompt_clean = render_open(item, order, clean_of[(seed, item_id)].replace(LETTER_SLOT, incorrect))
            answers.append((seed, "strength 0.5", generate_anchored(prompt_full, prompt_clean, 0.5)))
            answers.append((seed, "strength 1", generate_anchored(prompt_full, prompt_clean, 1)))
        for seed, name, text in answers:
            letter = letter_of(text)
            rows.append({"tagger training": seed, "item_id": item_id, "order": order, "answer from": name, "text": text, "letter": letter,
                         "result": "correct" if letter == correct else "incorrect" if letter == incorrect else "no letter"})
    if n % 25 == 0:
        print(n, "of", len(items), "| texts written:", len(written))
        pd.DataFrame(rows).to_csv(OUT_DIR / "A3_texts.csv", index=False)
texts = pd.DataFrame(rows)
texts.to_csv(OUT_DIR / "A3_texts.csv", index=False)
print("Saved:", OUT_DIR / "A3_texts.csv")


## 4. Criteria and results

In [ ]:
ORDER = ["no context", "evidence sentence", "strength 0", "strength 0.5", "strength 1"]


def verdict(judged, met):
    return "not judged" if not judged else "met" if met else "not met"


summaries, criteria_rows = [], []
for seed in SEEDS:
    part = texts[texts["tagger training"].isin(["all", seed])]
    summary = (part.groupby("answer from")["result"].value_counts(normalize=True).unstack(fill_value=0)
               .reindex(index=ORDER, columns=["correct", "incorrect", "no letter"], fill_value=0).mul(100))
    summary["answers"] = part.groupby("answer from").size().reindex(ORDER)
    summary.insert(0, "tagger training", seed)
    summaries.append(summary.reset_index())
    evidence, full, anchored = (summary.loc[name, "correct"] for name in ("evidence sentence", "strength 0", "strength 1"))
    recovered = (anchored - full) / (evidence - full) if evidence != full else float("nan")
    is_correct = part.assign(ok=part["result"] == "correct").pivot(index=["item_id", "order"], columns="answer from", values="ok")
    broken = (is_correct["strength 0"] & ~is_correct["strength 1"]).mean()
    judged = bool(evidence - full >= 5)
    criteria_rows.append({
        "tagger training": seed, "correct, evidence sentence alone (%)": evidence, "correct at strength 0 (%)": full,
        "correct at strength 0.5 (%)": summary.loc["strength 0.5", "correct"], "correct at strength 1 (%)": anchored,
        "8. share of the drop recovered at strength 1 (%)": 100 * recovered, "8.": verdict(judged, judged and recovered >= THRESHOLD),
        "9. correct at strength 0 and not at strength 1 (%)": 100 * broken, "9.": verdict(True, broken <= 0.05)})
summary_all = pd.concat(summaries)
summary_all.to_csv(OUT_DIR / "A3_summary.csv", index=False)
criteria = pd.DataFrame(criteria_rows)
criteria.to_csv(OUT_DIR / "A3_criteria.csv", index=False)
print(f"Items: {items['source_item_id'].nunique()} | answers for each row: {2 * len(items)}")
print("Criteria, by training of the tagger:")
display(criteria.round(1).T)
print("Share of answers (%) that start with the correct letter, the incorrect letter, or no letter:")
display(summary_all.round(1))
